# Lección 5 + 6 · Redes, y tu red de turnos de Siigo

**Qué vas a aprender**
- Qué es una **red**: nodos, arcos, costos y capacidades.
- El **problema de transporte** y el **flujo de costo mínimo**.
- **Conservación de flujo**: lo que entra a un nodo, sale.
- Por qué los modelos de redes salen enteros: la **matriz de incidencia** es TU.
- **Precio nodal**: cuánto cuesta una tonelada más en cada punto de la red.
- **Camino más corto**: un camino es una unidad de flujo.
- **Red de tiempo**: el turno de una persona es un camino. Así era tu modelo de Siigo.
- Del horario a la contratación: la **demanda no atendida** dice cuánta gente falta.
- Escribir cada modelo en **notación matemática** y leerla símbolo por símbolo.

**Cómo usar este notebook**
- Es doble: la primera mitad es redes (lección 5); la segunda, tu red de turnos (lección 6).
- Corre las celdas en orden con `Shift + Enter`.
- Antes de cada celda, adivina qué va a imprimir.

## 0. Preparar el entorno
Igual que en las lecciones anteriores. `networkx` (dibujar y recorrer redes) ya viene en Colab.

In [ ]:
%pip install -q pyomo highspy networkx

In [ ]:
import io
import time
from collections import defaultdict

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pyomo.environ as pyo

SOLVER = pyo.SolverFactory("appsi_highs")


def eur(x):
    return f"{x:,.0f}".replace(",", ".") + " €"

## 1. Dónde quedamos
En la lección 4 viste que la asignación sale entera sin branch & bound: su matriz es **totalmente unimodular** (TU).
Y quedó una promesa: transporte, flujo en redes y camino más corto tienen la misma suerte.

Hoy la cumplimos. Y llegamos a tu modelo de Siigo, que era una red.

**Analogía.** Un *rondo* (rotonda) de Cracovia. Los carros que entran son los mismos que salen. Ninguno aparece ni desaparece en el medio.
Esa regla, aplicada a cada punto de una red, es casi todo el modelo.

## 2. Transporte: dos almacenes, tres parques eólicos
Una empresa mantiene parques eólicos en el mar Báltico. Los repuestos salen de dos almacenes.
Hay que llevar a cada parque lo que pide esta semana, al menor costo.

| Costo (€ por tonelada) | parque_1 | parque_2 | parque_3 | Oferta (t) |
|---|---|---|---|---|
| **Gdansk** | 400 | 600 | 900 | 60 |
| **Szczecin** | 800 | 500 | 300 | 50 |
| **Demanda (t)** | 30 | 40 | 25 | |

**Las 3 piezas:**
- **Decido:** cuántas toneladas mando de cada almacén a cada parque.
- **Objetivo:** minimizar el costo de transporte.
- **Restricciones:** ningún almacén manda más de lo que tiene; cada parque recibe lo que pide.

### En notación
Conjuntos: $I$ = almacenes, $J$ = parques. Datos: $c_{ij}$ = costo por tonelada, $s_i$ = oferta, $d_j$ = demanda.
Variable: $x_{ij}$ = toneladas de $i$ a $j$.

$$
\begin{aligned}
\min \quad & \sum_{i \in I} \sum_{j \in J} c_{ij}\, x_{ij} \\
\text{s.a.} \quad & \sum_{j \in J} x_{ij} \le s_i && \forall\, i \in I && \text{(oferta)} \\
& \sum_{i \in I} x_{ij} = d_j && \forall\, j \in J && \text{(demanda)} \\
& x_{ij} \ge 0 && \forall\, i \in I,\ j \in J
\end{aligned}
$$

Es la última fila de la tabla de variantes de la lección 4, con $k_i = s_i$ y $n_j = d_j$.
Fíjate: $x_{ij} \ge 0$, **continua**. Las toneladas pueden tener decimales. Nadie pide enteros.

In [ ]:
OFERTA = {"Gdansk": 60, "Szczecin": 50}
DEMANDA = {"parque_1": 30, "parque_2": 40, "parque_3": 25}
COSTO = pd.DataFrame([[400, 600, 900],
                      [800, 500, 300]], index=list(OFERTA), columns=list(DEMANDA))


def transporte(oferta=OFERTA, demanda=DEMANDA, costo=COSTO):
    m = pyo.ConcreteModel()
    m.I = pyo.Set(initialize=list(oferta))
    m.J = pyo.Set(initialize=list(demanda))
    m.x = pyo.Var(m.I, m.J, within=pyo.NonNegativeReals)
    m.obj = pyo.Objective(expr=sum(costo.loc[i, j] * m.x[i, j] for i in m.I for j in m.J), sense=pyo.minimize)
    m.oferta = pyo.Constraint(m.I, rule=lambda m, i: sum(m.x[i, j] for j in m.J) <= oferta[i])
    m.demanda = pyo.Constraint(m.J, rule=lambda m, j: sum(m.x[i, j] for i in m.I) == demanda[j])
    m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    SOLVER.solve(m)
    return m


mt = transporte()
print("Costo:", eur(pyo.value(mt.obj)))
pd.DataFrame([[pyo.value(mt.x[i, j]) + 0.0 for j in mt.J] for i in mt.I], index=list(mt.I), columns=list(mt.J))

**Lectura:**
- Gdansk manda 30 t al parque 1 y 15 t al parque 2. Szczecin manda 25 t al parque 2 y 25 t al parque 3. Costo: **41.000 €**.
- **Todos los números son enteros**, y las variables eran continuas. Es la matriz TU de la lección 4:
  con ofertas y demandas enteras, las esquinas son enteras.
- Szczecin manda todo lo que tiene (50 t). Gdansk manda 45 de 60: le sobran 15 t.

Los precios sombra cuentan lo mismo en euros:

In [ ]:
pd.DataFrame({"restriccion": [f"oferta[{i}]" for i in mt.I] + [f"demanda[{j}]" for j in mt.J],
              "precio_sombra": [mt.dual[mt.oferta[i]] for i in mt.I] + [mt.dual[mt.demanda[j]] for j in mt.J]})

**Lectura:**
- `oferta[Szczecin]` = −100: una tonelada más en Szczecin **baja** el costo 100 €. Sale negativo porque minimizamos: tener más recurso solo puede bajar el costo.
- `oferta[Gdansk]` = 0: le sobran 15 t. Holgura complementaria, lección 2.
- `demanda[parque_2]` = 600: la tonelada más cara de servir. Guarda esta idea para la sección 4.

## 3. Una red de verdad: almacenes, bases y parques
El transporte directo es caro. En la vida real los repuestos van en camión a una **base portuaria** y de ahí en barco al parque.
Ahora hay 4 parques y 3 bases. Cada ruta tiene un costo por tonelada y una **capacidad** (el camión o el barco tiene un tamaño).

**Vocabulario de redes:**
- **Nodo**: un punto. Almacén (de donde sale), base (de paso) o parque (a donde llega).
- **Arco**: una conexión con dirección, de un nodo a otro. Tiene costo y capacidad.
- **Flujo**: cuánto pasa por cada arco. Es la variable de decisión.
- Un nodo **de paso** (*transbordo*) no produce ni consume: lo que entra, sale. El *rondo*.

Dos tablas, como en SQL: una de nodos y una de arcos. (Sin letras polacas en el código: `Leba` es Łeba.)

In [ ]:
df_nodos = pd.read_csv(io.StringIO('''nodo,tipo,oferta,demanda
Gdansk,almacen,60,0
Szczecin,almacen,70,0
Wladyslawowo,base,0,0
Leba,base,0,0
Ustka,base,0,0
parque_1,parque,0,30
parque_2,parque,0,40
parque_3,parque,0,25
parque_4,parque,0,15
'''))

df_arcos = pd.read_csv(io.StringIO('''origen,destino,medio,costo,capacidad,horas
Gdansk,Wladyslawowo,camion,100,60,1.5
Gdansk,Leba,camion,200,40,2.5
Szczecin,Ustka,camion,200,50,3.0
Szczecin,Leba,camion,300,30,4.0
Gdansk,parque_1,barco,600,20,9.0
Szczecin,parque_4,barco,700,20,10.0
Wladyslawowo,parque_1,barco,200,40,3.0
Wladyslawowo,parque_2,barco,400,30,4.5
Leba,parque_2,barco,200,30,3.5
Leba,parque_3,barco,300,30,4.0
Ustka,parque_3,barco,300,30,4.5
Ustka,parque_4,barco,200,30,3.0
'''))

print("Nodos:", len(df_nodos), "| arcos:", len(df_arcos))
df_arcos.head()

In [ ]:
COLUMNA = {"almacen": 0, "base": 1, "parque": 2}


def posiciones(df_nodos):
    pos = {}
    for tipo, grupo in df_nodos.groupby("tipo"):
        for k, n in enumerate(grupo["nodo"]):
            pos[n] = (COLUMNA[tipo], -k + (len(grupo) - 1) / 2)
    return pos


def dibujar(df_nodos, df_arcos, flujo=None, titulo=""):
    G = nx.DiGraph()
    G.add_nodes_from(df_nodos["nodo"])
    G.add_edges_from(zip(df_arcos["origen"], df_arcos["destino"]))
    pos = posiciones(df_nodos)
    fig, ax = plt.subplots(figsize=(9, 5))
    if flujo is None:
        anchos, etiquetas = 1.5, {(r.origen, r.destino): f"{r.costo} €/t" for r in df_arcos.itertuples()}
    else:
        anchos = [0.5 + 0.12 * flujo.get(a, 0) for a in G.edges]
        etiquetas = {a: f"{flujo[a]:.0f} t" for a in G.edges if flujo.get(a, 0) > 1e-6}
    nx.draw_networkx(G, pos, ax=ax, node_color="#dbeafe", edgecolors="#2563eb", node_size=2600, font_size=7,
                     width=anchos, arrowsize=14, edge_color="#64748b")
    nx.draw_networkx_edge_labels(G, pos, etiquetas, ax=ax, font_size=7, label_pos=0.35)
    ax.set_title(titulo)
    ax.axis("off")
    plt.show()


dibujar(df_nodos, df_arcos, titulo="La red: costo por tonelada en cada arco")

### 3.1 El modelo, en notación
Una red se escribe $G = (N, A)$: $N$ son los nodos y $A \subseteq N \times N$ son los arcos.
Un arco es un par ordenado $(i, j)$: va **de** $i$ **a** $j$. El orden importa: $(i, j) \ne (j, i)$.

Datos: $c_{ij}$ = costo por tonelada, $u_{ij}$ = capacidad del arco, $s_i$ = oferta de un almacén, $d_i$ = demanda de un nodo
(0 en las bases). $O \subseteq N$ = almacenes.

Dos atajos para "los arcos que salen de $i$" y "los arcos que entran a $i$":

$$ \delta^+(i) = \{\, (i, j) \in A \,\} \qquad\qquad \delta^-(i) = \{\, (j, i) \in A \,\} $$

$\delta$ (delta minúscula) con $+$ es "sale"; con $-$, "entra". El modelo de **flujo de costo mínimo**:

$$
\begin{aligned}
\min \quad & \sum_{(i,j) \in A} c_{ij}\, x_{ij} \\
\text{s.a.} \quad & \sum_{a \in \delta^-(i)} x_a - \sum_{a \in \delta^+(i)} x_a = d_i && \forall\, i \in N \setminus O && \text{(conservación)} \\
& \sum_{a \in \delta^+(i)} x_a - \sum_{a \in \delta^-(i)} x_a \le s_i && \forall\, i \in O && \text{(oferta)} \\
& 0 \le x_{ij} \le u_{ij} && \forall\, (i, j) \in A && \text{(capacidad)}
\end{aligned}
$$

| Símbolo | Se lee | Significa | En el código |
|---|---|---|---|
| $G = (N, A)$ | "la red G, con nodos N y arcos A" | la red entera son dos tablas | `df_nodos`, `df_arcos` |
| $(i, j)$ | "el arco de i a j" | un par **ordenado** | una fila de `df_arcos` |
| $\delta^+(i)$, $\delta^-(i)$ | "delta más de i", "delta menos de i" | arcos que salen, arcos que entran | `sale[i]`, `entra[i]` (con `groupby`) |
| $N \setminus O$ | "N menos O" | todos los nodos que no son almacén | `tipo != "almacen"` |
| $0 \le x_{ij} \le u_{ij}$ | "x entre 0 y u" | dos desigualdades en una línea | `bounds=(0, u)` |

**Lee la conservación en voz alta:** "lo que entra a $i$, menos lo que sale de $i$, es lo que $i$ se queda".
Base: se queda 0, el *rondo*. Parque: se queda su demanda.

In [ ]:
def flujo_costo_min(df_nodos, df_arcos):
    A = list(zip(df_arcos["origen"], df_arcos["destino"]))
    costo = dict(zip(A, df_arcos["costo"]))
    cap = dict(zip(A, df_arcos["capacidad"]))
    sale = df_arcos.groupby("origen")["destino"].apply(list).to_dict()    # δ⁺
    entra = df_arcos.groupby("destino")["origen"].apply(list).to_dict()   # δ⁻
    nodos = df_nodos.set_index("nodo")

    m = pyo.ConcreteModel()
    m.N = pyo.Set(initialize=list(nodos.index))
    m.A = pyo.Set(initialize=A, dimen=2)
    m.x = pyo.Var(m.A, bounds=lambda m, i, j: (0, cap[i, j]))
    m.obj = pyo.Objective(expr=sum(costo[a] * m.x[a] for a in m.A), sense=pyo.minimize)

    def balance(m, n):
        neto = sum(m.x[j, n] for j in entra.get(n, [])) - sum(m.x[n, j] for j in sale.get(n, []))  # entra − sale
        if nodos.loc[n, "tipo"] == "almacen":
            return -neto <= nodos.loc[n, "oferta"]
        return neto == nodos.loc[n, "demanda"]

    m.balance = pyo.Constraint(m.N, rule=balance)
    m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    SOLVER.solve(m)
    return m


mf = flujo_costo_min(df_nodos, df_arcos)
flujo = {a: pyo.value(mf.x[a]) for a in mf.A}
print("Costo:", eur(pyo.value(mf.obj)))
dibujar(df_nodos, df_arcos, flujo, titulo="Solución: toneladas por arco (más grueso = más flujo)")

In [ ]:
sol = df_arcos.assign(toneladas=[flujo[a] for a in zip(df_arcos["origen"], df_arcos["destino"])])
sol["lleno"] = sol["toneladas"] == sol["capacidad"]
sol[sol["toneladas"] > 0][["origen", "destino", "medio", "costo", "toneladas", "capacidad", "lleno"]]

**Lectura:**
- Costo: **45.500 €**. Ningún barco directo se usa: camión a una base más barco sale más barato.
- Gdansk manda sus 60 t. Szczecin manda 50 de 70: le sobran 20 t.
- Solo un arco está lleno: **Leba → parque_2** (30 de 30), la ruta barata al parque 2 (200 €/t).
  Las otras 10 t del parque 2 van por Wladyslawowo, a 400 €/t.
- Otra vez, todo entero sin pedirlo.

**La regla de la lección 4, en grande.** Con `groupby` armamos $\delta^+$ y $\delta^-$: cada restricción solo suma los arcos que existen.
Es el mismo truco de los pares factibles.

### 3.2 Por qué sale entero: la matriz de incidencia
Escribe la conservación como tabla: una fila por nodo, una columna por arco.
El arco $(i, j)$ pone **−1** en la fila de $i$ (sale) y **+1** en la fila de $j$ (entra).

In [ ]:
corto = lambda n: n[:4]
incidencia = pd.DataFrame(0, index=df_nodos["nodo"], columns=[f"{corto(i)}→{corto(j)}" for i, j in mf.A])
for k, (i, j) in enumerate(mf.A):
    incidencia.iloc[incidencia.index.get_loc(i), k] = -1
    incidencia.iloc[incidencia.index.get_loc(j), k] = 1
incidencia

**Lo que tiene de especial:** cada columna tiene exactamente un $-1$ y un $+1$. Nada más.
Una matriz así (la **matriz de incidencia nodo-arco** de una red con dirección) siempre es TU.

En la lección 4 cada columna tenía dos $+1$: uno en la fila del ingeniero y otro en la del proyecto.
Es el mismo patrón. La asignación **es** una red: ingenieros a la izquierda, proyectos a la derecha, un arco por par factible.

**En notación.** Si $E$ es la matriz de incidencia, toda la conservación se escribe en una línea:

$$ E\, x = d $$

y cada columna cumple $\sum_{i \in N} E_{i,a} = (+1) + (-1) = 0$ para todo arco $a \in A$: el flujo no aparece ni desaparece.
Con $d$ y $u$ enteros, todas las esquinas son enteras.

## 4. Precio nodal: cuánto vale una tonelada en cada punto
El precio sombra de la conservación de flujo de un nodo tiene un nombre: **precio nodal** (o *potencial* del nodo).
Dice cuánto sube el costo total si ese nodo pide **una tonelada más**.

**En notación**, con $\pi_i$ ("pi sub i") = precio sombra de la conservación del nodo $i$:

$$ \pi_i = \frac{\Delta z^*}{\Delta d_i} $$

**Analogía del sector.** En los mercados eléctricos con precios nodales (PJM en Estados Unidos, por ejemplo), el precio de la electricidad
cambia de un nodo de la red a otro. Ese precio es **exactamente este número**: el precio sombra de la conservación de flujo en el nodo,
con las líneas de transmisión como arcos con capacidad. Si una línea se llena, el precio de los nodos detrás de ella sube.

In [ ]:
tipo = df_nodos.set_index("nodo")["tipo"]
# La restricción del almacén está escrita al revés (sale − entra ≤ oferta): por eso su precio lleva el signo cambiado.
precios = pd.DataFrame({"nodo": list(mf.N), "tipo": [tipo[n] for n in mf.N],
                        "precio_nodal": [(-1 if tipo[n] == "almacen" else 1) * mf.dual[mf.balance[n]] + 0.0 for n in mf.N]})
precios

**Lectura:**
- **Parques:** el parque 2 es el más caro de servir (600 €/t). El parque 3 vale 500; los parques 1 y 4, 400.
- **Bases:** poner 1 t en Wladyslawowo vale 200; en Leba, 300; en Ustka, 200.
- **Almacenes:** Szczecin vale 0 porque le sobran 20 t. Gdansk vale 100: está lleno, y 1 t más ahí ahorra 100 €.

**La regla que arma los precios.** En cada arco **usado y no lleno**, el precio del destino es el del origen más el costo del arco:

$$ \pi_j = \pi_i + c_{ij} $$

- Ustka → parque_3: $500 = 200 + 300$. ✓
- Wladyslawowo → parque_2: $600 = 200 + 400$. ✓ La tonelada extra del parque 2 va por aquí.

En el arco **lleno** la regla se rompe: Leba → parque_2 da $600 - 300 = 300$, pero el arco cuesta 200.
Esos 100 € de diferencia son el **precio de la congestión**: lo que valdría 1 t más de capacidad en ese barco.

**Analogía del sector, otra vez.** Así se forman los precios nodales de la electricidad: precio = energía + pérdidas + **congestión**.
Cuando una línea se llena, el nodo de detrás se vuelve caro.

**Para el jefe de logística:** el precio nodal es el costo marginal de servir cada punto. Sirve para cobrar internamente,
para decidir dónde poner un almacén nuevo y para saber qué ruta ampliar.

### Comprobarlo
El parque 3 pide una tonelada más. Según su precio nodal, el costo debería subir 500 €.

In [ ]:
base = pyo.value(mf.obj)
otra = df_nodos.copy()
otra.loc[otra["nodo"] == "parque_3", "demanda"] += 1
nueva = pyo.value(flujo_costo_min(otra, df_arcos).obj)
print(f"Con 1 t más en parque_3: {eur(nueva)} (sube {eur(nueva - base)})")

## 5. Camino más corto: un camino es una unidad de flujo
Un técnico tiene que ir de Szczecin al parque 2 lo más rápido posible. La columna `horas` dice cuánto tarda cada arco.

No hace falta un modelo nuevo. Es flujo de costo mínimo con **una sola unidad**:
sale 1 de Szczecin, llega 1 al parque 2, y el costo de cada arco son sus horas.

**En notación**, con origen $s$ y destino $t$:

$$
\begin{aligned}
\min \quad & \sum_{(i,j) \in A} h_{ij}\, x_{ij} \\
\text{s.a.} \quad & \sum_{a \in \delta^+(i)} x_a - \sum_{a \in \delta^-(i)} x_a =
\begin{cases} 1 & \text{si } i = s \\ -1 & \text{si } i = t \\ 0 & \text{en otro caso} \end{cases} && \forall\, i \in N \\
& x_{ij} \ge 0 && \forall\, (i, j) \in A
\end{aligned}
$$

- La llave dice el lado derecho según el nodo: el origen da 1, el destino recibe 1, los demás son de paso.
- Otra vez $x_{ij} \ge 0$, continua. Por la matriz TU sale 0 o 1: cada arco se usa o no.
- Los arcos con $x_{ij} = 1$ forman el **camino**: una lista de nodos $(s, i_1, i_2, \dots, t)$ donde cada par seguido es un arco.

In [ ]:
def camino_mas_corto(df_arcos, s, t, peso="horas"):
    A = list(zip(df_arcos["origen"], df_arcos["destino"]))
    h = dict(zip(A, df_arcos[peso]))
    N = sorted(set(df_arcos["origen"]) | set(df_arcos["destino"]))
    b = {n: 1 if n == s else -1 if n == t else 0 for n in N}

    m = pyo.ConcreteModel()
    m.A = pyo.Set(initialize=A, dimen=2)
    m.x = pyo.Var(m.A, within=pyo.NonNegativeReals)
    m.obj = pyo.Objective(expr=sum(h[a] * m.x[a] for a in m.A), sense=pyo.minimize)
    m.balance = pyo.Constraint(N, rule=lambda m, n: sum(m.x[a] for a in A if a[0] == n) - sum(m.x[a] for a in A if a[1] == n) == b[n])
    SOLVER.solve(m)
    usados = [a for a in A if pyo.value(m.x[a]) > 0.5]
    print("Valores de x distintos de 0:", sorted({round(pyo.value(m.x[a]), 3) for a in A} - {0}))
    return usados, pyo.value(m.obj)


usados, horas = camino_mas_corto(df_arcos, "Szczecin", "parque_2")
print("Pyomo:   ", usados, "→", horas, "h")

G = nx.DiGraph()
G.add_weighted_edges_from(zip(df_arcos["origen"], df_arcos["destino"], df_arcos["horas"]), weight="horas")
print("networkx:", nx.shortest_path(G, "Szczecin", "parque_2", weight="horas"), "→",
      nx.shortest_path_length(G, "Szczecin", "parque_2", weight="horas"), "h")

**Lectura:**
- **Szczecin → Leba → parque 2: 7,5 h** (4 h de camión y 3,5 h de barco). Pyomo y `networkx` dan el mismo camino.
- La única $x$ distinta de 0 vale exactamente 1: el LP salió entero sin pedirlo. La matriz TU otra vez.
- `networkx` usa el **algoritmo de Dijkstra**: un algoritmo especial, como el método húngaro de la lección 4. Muy rápido, pero no acepta reglas extra.

**La idea que lleva a la segunda mitad:**
> Una unidad de flujo que va de un origen a un destino **dibuja un camino**. Si los nodos son momentos del día,
> el camino es un **horario**.

Eso era tu modelo de Siigo.

# Segunda parte · Lección 6: tu red de turnos

## 6. La red de tiempo: un turno es un camino
En Siigo cada agente tenía su propia red. Una unidad de flujo iba del `inicio` al `Fin`, y **el camino que recorría era su horario**.

**Analogía.** Un tablero de juego por casillas. Cada casilla es "qué hago en esta media hora": trabajar, descansar, almorzar.
Las flechas dicen a qué casilla puedes pasar. Si no hay flecha, ese movimiento es ilegal. Cualquier recorrido del tablero es un horario legal.

**Las reglas de esta versión de estudio** (bloques de 30 min, de 07:00 a 18:00):
- **Entrada:** 07:00, 07:30, 08:00 u 08:30.
- **Descanso:** un bloque de 30 min que empieza entre 08:30 y 10:00.
- **Almuerzo:** 1 hora (2 bloques) que empieza a las 12:00, 12:30, 13:00 o 13:30.
- **Horas:** 8 h de trabajo por día (16 bloques). Con almuerzo y descanso: 9,5 h en la empresa. Entra 07:00 → sale 16:30; entra 08:30 → sale 18:00.
- **Etapas:** el trabajo del día se parte en 3 etapas: antes del descanso, entre descanso y almuerzo, después del almuerzo.
  Solo se pasa de una etapa a la siguiente **por** un descanso o **por** el almuerzo. Así nadie se los salta.
- **Días libres conocidos:** ese día la red solo tiene el nodo `libre`.

**Diferencia con Siigo.** Allá eran 2 descansos de 15 min, con dos nodos por bloque (`08:30_30` y `08:30_45`), y 5 países.
Aquí un descanso de 30 min para que la red de un día quepa en una pantalla. La técnica es la misma.

In [ ]:
BLOQUES = list(range(22))         # bloque 0 = 07:00–07:30 … bloque 21 = 17:30–18:00
ENTRADA = [0, 1, 2, 3]             # 07:00 a 08:30
DESCANSO = [3, 4, 5, 6]            # empieza 08:30 a 10:00
ALMUERZO = [10, 11, 12, 13]        # empieza 12:00 a 13:30, dura 2 bloques
BLOQUES_TRABAJO = 16               # 8 h


def hora(t):
    minutos = 7 * 60 + 30 * t
    return f"{minutos // 60:02d}:{minutos % 60:02d}"


print("Bloques:", hora(BLOQUES[0]), "…", hora(BLOQUES[-1] + 1), "| entrada:", [hora(t) for t in ENTRADA])

### 6.1 Nodos y arcos, en notación
Un nodo es una **tupla** de 4 datos: $n = (p, d, \text{tipo}, t)$ = persona, día, qué hace, bloque.
Por ejemplo, $(\text{Andrés}, \text{lun}, \text{trabajo}_1, 9)$ = "Andrés, el lunes, trabajando en la etapa 1, de 11:30 a 12:00".
Entre un día y el siguiente hay un nodo `corte`. El corte antes del lunes es el `inicio` $s_p$; el corte después del viernes es el `Fin` $f_p$.

Los arcos de un día, con $e$ = etapa:

| Arco | De | A | Cuándo existe |
|---|---|---|---|
| entrada | corte del día $d$ | $(\text{trabajo}_0, t)$ | $t \in$ ENTRADA |
| seguir trabajando | $(\text{trabajo}_e, t)$ | $(\text{trabajo}_e, t+1)$ | dentro de la etapa $e$ |
| ir a descanso | $(\text{trabajo}_0, t-1)$ | $(\text{descanso}, t)$ | $t \in$ DESCANSO |
| volver del descanso | $(\text{descanso}, t)$ | $(\text{trabajo}_1, t+1)$ | siempre |
| almorzar | $(\text{trabajo}_1, t-1) \to (\text{almuerzo}, t) \to (\text{almuerzo}_2, t+1)$ | $(\text{trabajo}_2, t+2)$ | $t \in$ ALMUERZO |
| salida | $(\text{trabajo}_2, t)$ | corte del día $d+1$ | después del almuerzo |
| día libre | corte del día $d$ $\to$ libre | corte del día $d+1$ | solo si el día libre es conocido |

La red de la persona $p$ es $G_p = (N_p, A_p)$. Para un día laboral, el código la genera así:

In [ ]:
def red_dia(p, d, libre=False):
    # Arcos de la persona p en el día d. Nodo = (persona, día, tipo, bloque).
    ini, fin = (p, d, "corte", None), (p, d + 1, "corte", None)
    if libre:  # solo red factible: un día libre conocido no tiene nodos de trabajo
        L = (p, d, "libre", None)
        return [(ini, L), (L, fin)]

    def T(e, t):
        return (p, d, f"trabajo{e}", t)

    arcos = [(ini, T(0, t)) for t in ENTRADA]                                               # entrada
    arcos += [(T(0, t), T(0, t + 1)) for t in range(0, max(DESCANSO) - 1)]                  # etapa 0
    for t in DESCANSO:                                                                      # descanso
        arcos += [(T(0, t - 1), (p, d, "descanso", t)), ((p, d, "descanso", t), T(1, t + 1))]
    arcos += [(T(1, t), T(1, t + 1)) for t in range(min(DESCANSO) + 1, max(ALMUERZO) - 1)]  # etapa 1
    for t in ALMUERZO:                                                                      # almuerzo: 2 bloques
        a1, a2 = (p, d, "almuerzo", t), (p, d, "almuerzo2", t + 1)
        arcos += [(T(1, t - 1), a1), (a1, a2), (a2, T(2, t + 2))]
    arcos += [(T(2, t), T(2, t + 1)) for t in range(min(ALMUERZO) + 2, max(BLOQUES))]       # etapa 2
    arcos += [(T(2, t), fin) for t in range(min(ALMUERZO) + 2, max(BLOQUES) + 1)]           # salida
    return arcos


dia = red_dia("Andrés", 0)
G_dia = nx.DiGraph(dia)
print("Un día de Andrés:", G_dia.number_of_nodes(), "nodos y", G_dia.number_of_edges(), "arcos")

In [ ]:
FILA = {"trabajo0": 4, "descanso": 3, "trabajo1": 2, "almuerzo": 1.2, "almuerzo2": 0.8, "trabajo2": 0, "corte": 2}
COLOR = {"trabajo0": "#dbeafe", "trabajo1": "#dbeafe", "trabajo2": "#dbeafe", "descanso": "#dcfce7",
         "almuerzo": "#ffedd5", "almuerzo2": "#ffedd5", "corte": "#1f2937", "libre": "#f3f4f6"}


def pos_nodo(n):
    p, d, tipo, t = n
    if tipo == "corte":
        return (-1.5 if d == 0 else 23.5, FILA["corte"])
    return (t, FILA[tipo])


def dibujar_dia(G, camino=None):
    fig, ax = plt.subplots(figsize=(14, 4.5))
    pos = {n: pos_nodo(n) for n in G.nodes}
    resaltar = set(zip(camino, camino[1:])) if camino else set()
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color=["#dc2626" if e in resaltar else "#cbd5e1" for e in G.edges],
                           width=[2.5 if e in resaltar else 0.8 for e in G.edges], arrowsize=8, node_size=120)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=120, node_color=[COLOR[n[2]] for n in G.nodes], edgecolors="#475569")
    ax.set_xticks(BLOQUES[::2], [hora(t) for t in BLOQUES[::2]], fontsize=8)
    ax.set_yticks([4, 3, 2, 1, 0], ["trabaja · etapa 0", "descanso", "trabaja · etapa 1", "almuerzo", "trabaja · etapa 2"])
    ax.tick_params(left=False, bottom=False, labelleft=True, labelbottom=True)
    ax.set_title("Red de un día: corte (negro, izquierda) → … → corte (negro, derecha)")
    plt.show()


dibujar_dia(G_dia)

**Cómo se lee el dibujo:**
- Cada punto es un nodo: una media hora haciendo algo. Las columnas son las horas.
- Azul = trabajar, verde = descanso, naranja = almuerzo. Los dos puntos negros son los cortes entre días.
- Para bajar de una fila azul a la siguiente **hay que pasar** por verde o por naranja. No hay otra flecha.

### 6.2 Fuerza bruta: todos los caminos de un día
Con `networkx` podemos listar **todos** los caminos de un corte al siguiente. Cada uno es un horario.

In [ ]:
ini, fin = ("Andrés", 0, "corte", None), ("Andrés", 1, "corte", None)
caminos = list(nx.all_simple_paths(G_dia, ini, fin))
trabajo = lambda camino: sum(n[2].startswith("trabajo") for n in camino)
legales = [c for c in caminos if trabajo(c) == BLOQUES_TRABAJO]


def horario(camino):
    pasos = [(n[2], n[3]) for n in camino if n[3] is not None]
    entrada = hora(pasos[0][1])
    descanso = hora(next(t for tipo, t in pasos if tipo == "descanso"))
    almuerzo = hora(next(t for tipo, t in pasos if tipo == "almuerzo"))
    salida = hora(pasos[-1][1] + 1)
    return f"entra {entrada} · descanso {descanso} · almuerzo {almuerzo} · sale {salida}"


print("Caminos de corte a corte:", len(caminos))
print("Con exactamente 8 h de trabajo:", len(legales))
print("Horas de trabajo posibles sin la regla:", sorted({trabajo(c) / 2 for c in caminos}))
for c in legales[:3]:
    print("  ", horario(c))
dibujar_dia(G_dia, legales[0])

**Lectura:**
- La red tiene 510 caminos. **Todos** respetan entrada, descanso, almuerzo y etapas: un horario ilegal no es un camino.
- Pero la red sola no fija las horas: hay caminos de 3,5 a 9,5 h. La regla de 8 h deja **60 horarios legales**:
  15 combinaciones de entrada y descanso (el descanso va después de entrar) × 4 almuerzos.
- Con 8 h fijas, la hora de salida queda fija: entra 07:00 → sale 16:30.
- Esa regla no está en la red. Es una **restricción extra** (*side constraint*), como en Siigo, donde los costos de los arcos eran horas
  y una restricción fijaba las horas del día.

**Lo que puedes decir en la entrevista:** *"The network encodes the labor rules: illegal shifts are not paths, so I don't need one constraint per rule."*

## 7. El equipo, una semana
12 personas, de lunes a viernes. Tres tablas, como en SQL:
1. **Personas.**
2. **Días libres conocidos:** ese día, esa persona solo tiene el nodo `libre`.
3. **Demanda:** cuántas personas hacen falta trabajando en cada bloque de cada día. Dos picos: media mañana y media tarde.

La demanda es simulada con semilla fija: siempre sale igual.

In [ ]:
DIAS = ["lun", "mar", "mie", "jue", "vie"]
PERSONAS = ["Andrés", "Klaudia", "Oliver", "Marta", "Piotr", "Lucía",
            "Tomasz", "Valentina", "Kuba", "Sofía", "Wiktor", "Camila"]
df_libres = pd.DataFrame([("Andrés", "mie"), ("Klaudia", "vie"), ("Oliver", "lun"), ("Marta", "jue"),
                          ("Lucía", "mar"), ("Kuba", "vie"), ("Sofía", "lun"), ("Camila", "mie")],
                         columns=["persona", "dia"])

rng = np.random.default_rng(3)
perfil = np.array([2, 3, 5, 7, 8, 9, 9, 8, 8, 7, 6, 5, 5, 6, 7, 8, 8, 7, 6, 4, 3, 2])  # forma de un día típico
factor = {"lun": 1.15, "mar": 1.0, "mie": 0.95, "jue": 1.0, "vie": 0.85}
df_demanda = pd.DataFrame([(d, t, int(max(0, round(perfil[t] * factor[d] + rng.normal(0, 0.6)))))
                           for d in DIAS for t in BLOQUES], columns=["dia", "bloque", "demanda"])

print("Personas-día que trabajan:", len(PERSONAS) * len(DIAS) - len(df_libres))
print("Bloques-persona disponibles:", (len(PERSONAS) * len(DIAS) - len(df_libres)) * BLOQUES_TRABAJO,
      "| demanda total:", df_demanda["demanda"].sum())
df_demanda.pivot(index="dia", columns="bloque", values="demanda").loc[DIAS].rename(columns=hora)

Hay más horas disponibles que demanda: 832 contra 657 bloques-persona. ¿Alcanza?
No necesariamente: la gente no está repartida a cualquier hora. Todos descansan a media mañana, justo en el pico.

### 7.1 El modelo completo, en notación
Conjuntos: $P$ = personas, $D$ = días, $T$ = bloques. La red de cada persona es $G_p = (N_p, A_p)$, y la red del equipo es la unión:

$$ A = \bigcup_{p \in P} A_p $$

$\bigcup$ ("unión") pega las listas de arcos de todas las personas. En Python: `arcos += red_dia(p, d)` dentro de dos `for`.

Subconjuntos de arcos que vamos a sumar:
- $W_{p,d} \subseteq A_p$ = arcos que **entran** a un nodo de trabajo de $p$ el día $d$. Cada uno es media hora trabajada.
- $W_{d,t} \subseteq A$ = arcos que entran a un nodo de trabajo en el bloque $t$ del día $d$, de cualquier persona.

Datos: $r_{d,t}$ = demanda del bloque. $L$ = pares (persona, día) libres conocidos.
Variables: $x_a \in \{0,1\}$ = 1 si el camino usa el arco $a$; $u_{d,t} \ge 0$ = personas que faltan en el bloque.

$$
\begin{aligned}
\min \quad & \sum_{d \in D} \sum_{t \in T} u_{d,t} \\
\text{s.a.} \quad & \sum_{a \in \delta^+(n)} x_a - \sum_{a \in \delta^-(n)} x_a =
\begin{cases} 1 & n = s_p \\ -1 & n = f_p \\ 0 & \text{en otro caso} \end{cases} && \forall\, p \in P,\ n \in N_p && \text{(un camino por persona)} \\
& \sum_{a \in W_{p,d}} x_a = 16 && \forall\, (p, d) \notin L && \text{(8 h por día)} \\
& \sum_{a \in W_{d,t}} x_a + u_{d,t} \ge r_{d,t} && \forall\, d \in D,\ t \in T && \text{(cobertura)} \\
& x_a \in \{0, 1\} && \forall\, a \in A \\
& u_{d,t} \ge 0 && \forall\, d \in D,\ t \in T
\end{aligned}
$$

- **Conservación:** cada persona es una unidad de flujo de su `inicio` a su `Fin`. Su camino es su semana.
- **Horas:** la *side constraint* de la sección 6.2. $\notin$ se lee "no pertenece a": solo los días que trabaja.
- **Cobertura:** la única restricción que **junta** a las personas. Quien está en descanso o almorzando no entra en $W_{d,t}$: no cuenta.
- **Elástica:** $u_{d,t}$ es la idea del consultor externo de la lección 4. El modelo nunca es infactible: dice dónde falta gente.

En Siigo los arcos además tenían costo en horas y la falta tenía un castigo. Como aquí las horas son fijas, solo queda la falta.

In [ ]:
def turnos(personas, df_libres, df_demanda, relajar=False):
    libres = set(zip(df_libres["persona"], df_libres["dia"].map(DIAS.index)))
    demanda = {(DIAS.index(d), t): r for d, t, r in df_demanda.itertuples(index=False)}

    arcos = []
    for p in personas:
        for d in range(len(DIAS)):
            arcos += red_dia(p, d, libre=(p, d) in libres)
    sale, entra = defaultdict(list), defaultdict(list)        # δ⁺ y δ⁻ de cada nodo
    W_pd, W_dt = defaultdict(list), defaultdict(list)          # arcos de trabajo por (persona, día) y por (día, bloque)
    for k, (o, n) in enumerate(arcos):
        sale[o].append(k)
        entra[n].append(k)
        p, d, tipo, t = n
        if tipo.startswith("trabajo"):
            W_pd[p, d].append(k)
            W_dt[d, t].append(k)

    m = pyo.ConcreteModel()
    m.A = pyo.RangeSet(0, len(arcos) - 1)
    m.x = pyo.Var(m.A, within=pyo.UnitInterval if relajar else pyo.Binary)
    m.u = pyo.Var(list(demanda), within=pyo.NonNegativeReals)
    m.obj = pyo.Objective(expr=sum(m.u.values()), sense=pyo.minimize)

    def lado_derecho(n):
        p, d, tipo, t = n
        if tipo == "corte" and d == 0:
            return 1             # inicio s_p
        if tipo == "corte" and d == len(DIAS):
            return -1            # Fin f_p
        return 0

    nodos = list(set(sale) | set(entra))
    m.flujo = pyo.Constraint(range(len(nodos)), rule=lambda m, k: (
        sum(m.x[a] for a in sale[nodos[k]]) - sum(m.x[a] for a in entra[nodos[k]]) == lado_derecho(nodos[k])))
    m.horas = pyo.Constraint(list(W_pd), rule=lambda m, p, d: sum(m.x[a] for a in W_pd[p, d]) == BLOQUES_TRABAJO)
    m.cobertura = pyo.Constraint(list(demanda), rule=lambda m, d, t: (
        sum(m.x[a] for a in W_dt[d, t]) + m.u[d, t] >= demanda[d, t]))

    inicio = time.perf_counter()
    SOLVER.solve(m)
    m.segundos = time.perf_counter() - inicio
    m.arcos, m.nodos = arcos, nodos
    return m


mt7 = turnos(PERSONAS, df_libres, df_demanda)
print(f"Nodos: {len(mt7.nodos):,} | arcos (binarias): {len(mt7.arcos):,} | restricciones: "
      f"{len(mt7.flujo) + len(mt7.horas) + len(mt7.cobertura):,}".replace(",", "."))
print(f"Resuelto en {mt7.segundos:.1f} s | bloques-persona sin cubrir: {pyo.value(mt7.obj):.0f}")

### 7.2 El horario, de vuelta a tabla
Cada camino se traduce a una fila por persona: qué hace en cada bloque.

In [ ]:
LETRA = {"trabajo0": "T", "trabajo1": "T", "trabajo2": "T", "descanso": "D", "almuerzo": "A", "almuerzo2": "A", "libre": "libre"}


def horario_semana(m):
    filas = []
    for k, (o, n) in enumerate(m.arcos):
        if pyo.value(m.x[k]) > 0.5 and n[2] in LETRA:
            p, d, tipo, t = n
            filas.append({"persona": p, "dia": DIAS[d], "bloque": t, "hace": LETRA[tipo]})
    return pd.DataFrame(filas)


hs = horario_semana(mt7)
lunes = (hs[(hs["dia"] == "lun") & (hs["bloque"].notna())]
         .pivot(index="persona", columns="bloque", values="hace").fillna("·"))
lunes.columns = [hora(int(t)) for t in lunes.columns]
lunes  # Oliver y Sofía no aparecen: el lunes es su día libre

In [ ]:
def grafico_cobertura(m, df_demanda, dia="lun"):
    d = DIAS.index(dia)
    dem = df_demanda[df_demanda["dia"] == dia].set_index("bloque")["demanda"]
    trabajando = [sum(pyo.value(m.x[k]) > 0.5 for k, (o, n) in enumerate(m.arcos)
                      if n[1] == d and n[2].startswith("trabajo") and n[3] == t) for t in BLOQUES]
    falta = [pyo.value(m.u[d, t]) for t in BLOQUES]
    fig, ax = plt.subplots(figsize=(11, 3.5))
    ax.bar(BLOQUES, trabajando, color="#93c5fd", label="personas trabajando")
    ax.bar(BLOQUES, falta, bottom=trabajando, color="#f87171", label="falta (u)")
    ax.step(BLOQUES, dem.values, where="mid", color="#111827", label="demanda")
    ax.set_xticks(BLOQUES[::2], [hora(t) for t in BLOQUES[::2]], fontsize=8)
    tope = max(max(np.add(trabajando, falta)), dem.max()) + 1
    ax.set(ylabel="personas", ylim=(0, tope), title=f"Cobertura del {dia}")
    ax.legend(loc="upper right", fontsize=8)
    plt.show()


grafico_cobertura(mt7, df_demanda, "lun")
falta = pd.Series({(DIAS[d], hora(t)): round(pyo.value(v)) for (d, t), v in mt7.u.items() if pyo.value(v) > 0.5}, name="faltan")
falta.to_frame()

**Lectura para el jefe de soporte:**
- Faltan **12 bloques-persona** en la semana: 6 horas-persona. Es poco, pero está concentrado.
- 10 de los 12 caen entre **08:30 y 10:30**: la ventana del descanso, justo en el pico de la mañana.
  Todos tienen que descansar ahí, y cada descanso saca a una persona del pico.
- El **lunes** es el peor día (8 de 12): tiene la demanda más alta y Oliver y Sofía libres.
- Ese número no es un error del modelo: es información. Dice **dónde y cuánta** gente falta.
- Hay muchos horarios igual de buenos. Con otra versión de HiGHS el total (12) es el mismo, pero los bloques exactos pueden cambiar un poco.

### 7.3 ¿Por qué resuelve rápido?
Hay 2.928 binarias. En la lección 3 eso podía tardar. Miremos el LP relajado.

In [ ]:
mr = turnos(PERSONAS, df_libres, df_demanda, relajar=True)
fraccionales = sum(1e-6 < pyo.value(v) < 1 - 1e-6 for v in mr.x.values())
print(f"LP relajado: {pyo.value(mr.obj):.2f} en {mr.segundos:.2f} s | MILP: {pyo.value(mt7.obj):.2f} en {mt7.segundos:.2f} s")
print("Arcos fraccionales en el LP:", fraccionales)

**Lectura:**
- El LP tiene 391 arcos fraccionales: personas "partidas" entre dos horarios. Ya no es TU.
- Pero **vale lo mismo** que el MILP: la cota es exacta, el gap en la raíz es 0. El solver casi no ramifica.

**Por qué.** Quita las horas y la cobertura: cada persona sola es un camino más corto (matriz de incidencia, TU).
Solo esas dos restricciones juntan a las personas y meten fracciones. La estructura de red sigue ayudando,
como la asignación con una regla extra de la lección 4.

> *"Without the coverage and hours constraints, each agent is a shortest-path problem, which is totally unimodular.
> Only those side constraints couple the agents, so the LP relaxation stays very tight and the MILP solves fast."*

## 8. Del horario a la contratación
En Siigo, la demanda no atendida de un modelo **operativo** (el horario de la semana) servía para una decisión **táctica**: cuánta gente contratar.

Probemos: contratamos 1 persona nueva, sin días libres, y re-resolvemos. Es un MILP: la sensibilidad es **re-solve**, regla de la lección 3.

In [ ]:
mx = turnos(PERSONAS + ["Nueva_1"], df_libres, df_demanda)
print(f"Con 1 persona nueva: faltan {pyo.value(mx.obj):.0f} bloques-persona (antes {pyo.value(mt7.obj):.0f})")
grafico_cobertura(mx, df_demanda, "lun")

**Lectura:**
- La falta baja de 12 a **4** bloques-persona.
- La persona nueva trabaja 80 bloques en la semana (16 × 5), pero solo quita 8 de falta.
  Los otros 72 caen en horas donde ya había gente: la falta está en pocos bloques puntuales.

**Para el jefe:** *"Con el equipo actual faltan 6 horas-persona por semana, concentradas a media mañana.
Una persona más las reduce a 2 horas."* Y la pregunta siguiente es de negocio: ¿vale lo que cuesta?

## 9. Resumen: conceptos de hoy
| Concepto | Qué es | En la lección | En inglés |
|---|---|---|---|
| Red | Nodos y arcos con dirección | almacenes, bases, parques | network, directed graph |
| Transporte | Oferta y demanda, arcos directos | 2 almacenes × 3 parques | transportation problem |
| Flujo de costo mínimo | Mandar la demanda por la red al menor costo, con capacidades | 12 arcos, 45.500 € | minimum cost flow |
| Conservación de flujo | Lo que entra a un nodo menos lo que sale es lo que se queda | el *rondo* | flow conservation |
| Matriz de incidencia | Un −1 y un +1 por columna: siempre TU | todo sale entero | node-arc incidence matrix |
| Precio nodal | Precio sombra de la conservación de un nodo | parque 2: 600 €/t | node potential, LMP |
| Camino más corto | Flujo de costo mínimo de una unidad | Szczecin → parque 2 | shortest path |
| Red de tiempo | Nodos = momentos; camino = horario | tu red de Siigo | time-expanded network |
| Restricción extra | Regla que no cabe en la red y rompe la TU | 8 h por día, cobertura | side constraint |
| Cobertura elástica | Falta con castigo: nunca infactible | faltan 12 bloques | elastic coverage, unmet demand |

### Notación de hoy
| Símbolo | Significa | En la lección |
|---|---|---|
| $G = (N, A)$ | red: nodos y arcos | dos tablas |
| $(i, j)$ | arco: par **ordenado**, de $i$ a $j$ | Gdansk → Wladyslawowo |
| $\delta^+(i)$, $\delta^-(i)$ | arcos que salen de $i$, arcos que entran a $i$ | `sale[i]`, `entra[i]` |
| $N \setminus O$ | diferencia: N sin los elementos de O | nodos que no son almacén |
| $0 \le x_{ij} \le u_{ij}$ | cota inferior y superior en una línea | capacidad del barco |
| $E\,x = d$ | conservación de toda la red como matriz | matriz de incidencia |
| $\pi_i$ | precio sombra de la conservación: precio nodal | $\pi_{\text{parque 2}}$ |
| $(s, i_1, \dots, t)$ | camino: lista de nodos | Szczecin → Leba → parque 2 |
| $(p, d, \text{tipo}, t)$ | nodo como tupla de 4 datos | Andrés, lun, trabajo, 09:30 |
| $\bigcup_{p \in P} A_p$ | unión: pegar los conjuntos de todos | arcos del equipo |
| $\notin$ | no pertenece a | días que no son libres |

## 10. Ejercicios
1. **Tormenta.** Por el mal tiempo, el barco Leba → parque_2 solo puede llevar 10 t en vez de 30.
   Antes de resolver, predice cuánto sube el costo con el **precio de la congestión** de la sección 4. Después comprueba. ¿Qué rutas cambian?
2. **Descanso más temprano.** El sindicato acepta que el descanso empiece desde las 08:00. Cambia `DESCANSO = [2, 3, 4, 5, 6]` y re-resuelve el equipo.
   Antes, adivina: ¿la falta baja, sube o queda igual? ¿Por qué? Al final, vuelve a poner `DESCANSO = [3, 4, 5, 6]`.
3. **¿Cuántas personas contratar?** Con la función `turnos`, agrega 1, 2, 3… personas nuevas hasta que la falta llegue a 0.
   ¿Cuántas hacen falta? ¿Cuánto baja la falta con cada una? ¿Qué te dice que no baje parejo?

In [ ]:
# Escribe aquí tus intentos

## En la entrevista
Pregunta típica: *"Why do network flow models solve so fast, even with millions of variables?"*
> *"The node-arc incidence matrix has exactly one +1 and one −1 per column, so it is totally unimodular.
> With integer supplies, demands and capacities, the LP relaxation is already integral. On top of that there are
> specialized algorithms, like Dijkstra for shortest paths or network simplex for min-cost flow."*

Seguimiento: *"What does the dual of the flow conservation constraint mean?"*
> *"It is the node potential: the marginal cost of delivering one more unit at that node. In power markets
> with nodal pricing, the locational marginal price is exactly that dual."*

Y tu caso: *"Tell me about a model you built."* → la explicación de 80 segundos de `interview_prep/09_red_siigo.md`.
Ahora puedes dibujarla: una red por persona, camino = horario, reglas en la red, horas y cobertura como *side constraints*,
falta penalizada que dice cuánta gente contratar.

## Soluciones (no mires antes de intentar)

In [ ]:
# Ejercicio 1
congestion = precios.set_index("nodo").loc["parque_2", "precio_nodal"] - precios.set_index("nodo").loc["Leba", "precio_nodal"] - 200
print("Predicho:", eur(congestion), "por tonelada ×", 30 - 10, "t =", eur(congestion * 20))

tormenta = df_arcos.copy()
tormenta.loc[(tormenta["origen"] == "Leba") & (tormenta["destino"] == "parque_2"), "capacidad"] = 10
m1 = flujo_costo_min(df_nodos, tormenta)
print("Real:", eur(pyo.value(m1.obj)), "| sube", eur(pyo.value(m1.obj) - pyo.value(mf.obj)))
cambios = pd.DataFrame({"antes": flujo, "tormenta": {a: pyo.value(m1.x[a]) + 0.0 for a in m1.A}})
cambios[cambios["antes"] != cambios["tormenta"]]

Predicho: 100 €/t × 20 t = **2.000 €**. Real: lo mismo, porque 20 t están dentro del rango.
Las 20 t que ya no caben en Leba → parque_2 se van por Wladyslawowo → parque_2 (200 € más caro por tonelada).
Pero Gdansk ya no manda a Leba (200 €/t) sino a Wladyslawowo (100 €/t), y eso ahorra 100 €/t.
Neto: 100 €/t, el precio de la congestión. Una ruta llena se nota en el precio, igual que una línea eléctrica llena.

In [ ]:
# Ejercicio 2
DESCANSO = [2, 3, 4, 5, 6]
m2 = turnos(PERSONAS, df_libres, df_demanda)
print("Con descanso desde las 08:00, faltan:", round(pyo.value(m2.obj)), "| antes:", round(pyo.value(mt7.obj)))
DESCANSO = [3, 4, 5, 6]  # volver a la regla original

La falta baja de 12 a **7**. Con una ventana de descanso más ancha, los descansos se reparten en más bloques
y menos gente sale del pico de 09:00 a 10:00 al mismo tiempo.
La regla laboral más flexible vale 5 bloques-persona por semana. Es el "costo de una regla" de la lección 3, al revés:
ahora mides cuánto **ahorra** cambiarla. Con la red, cambiar una regla es cambiar una lista: el modelo no se toca.

In [ ]:
# Ejercicio 3
for nuevas in range(1, 6):
    equipo = PERSONAS + [f"Nueva_{k}" for k in range(1, nuevas + 1)]
    faltan = round(pyo.value(turnos(equipo, df_libres, df_demanda).obj))
    print(f"{nuevas} nueva(s): faltan {faltan}")
    if faltan == 0:
        break

Hacen falta **2 personas**. La primera quita 8 bloques; la segunda, los 4 que quedan.
No baja parejo porque la falta está en pocos bloques puntuales: cada persona de tiempo completo cubre sobre todo horas donde ya había gente.
Antes de contratar 2 personas, vale la pena probar otras salidas: el descanso más temprano del ejercicio 2,
o una persona de medio tiempo solo para la mañana. El modelo da los números; la forma de cubrir es una decisión de negocio.